**[Source]** Jisoo Project (`지수프젝/3_플래그_표본확인_v3b.ipynb`, 원본 SHA256 앞 16자리 `476e7263b0677ae5`)
**[Status]** FIXED
**[Role]** 품질 플래그 표본 수동 검토 (데이터 품질 분석)
**[Modification]** 원본 코드 수정 금지. 이 파일은 원본의 복사본이며 **맨 위에 이 안내 셀 1개만 추가**했다(코드·출력 셀은 원본과 동일, config/fixed_manifest.json의 code_sha256으로 확인).
**[Rerun]** 재실행하지 않는다. 이미 실행된 결과(`data/preprocessed_final`)를 05번 노트북에서 독립 검증한 뒤 사용한다.

# 플래그 표본 확인 노트북 (사람이 직접 True/False 판정)

`전처리_v3b.ipynb`에서 생성한 학습용 train 데이터의 품질 플래그는
"정답 오류 확정"이 아니라 "의심 사례 표시"이다.

본 노트북에서는 train 데이터에서 각 품질 플래그별 표본을 무작위 추출하고,
사람이 실제 target의 결함 여부를 판정하여
플래그가 실제 오류 데이터를 선별하는 데 유효한지 검토한다.

또한 품질 플래그가 없는 행을 통제군으로 표본 추출하여,
플래그 표본의 오류율과 비교한다.

**진행 순서**
1. 위에서부터 실행합니다 (설정 → 표본 뽑기).
2. 플래그별 표는 번호가 붙어 있습니다. 표를 보면서 **T(문제 있음)인 행의 번호와 판정 이유를** 아래 `BAD` 딕셔너리에 적습니다. 애매한 행은 `UNSURE`에 적습니다. 한 플래그의 표본을 모두 확인한 뒤, T와 ?가 하나도 없다면 해당 플래그를 `NONE_BAD`에 추가합니다.
3. 결과 셀을 실행하면 플래그별 T 비율과 95% 신뢰구간, "이 플래그로 행을 지우면 버려지는 정상 행 추정치"가 나옵니다.

**원칙**: 이 노트북은 데이터를 수정하지 않고 읽기만 합니다(train.jsonl은 그대로). 표본은 시드가 고정되어 다시 실행해도 같은 행이 나옵니다.

In [1]:
# 0. 설정
import os, json, random, math, datetime
from pathlib import Path
import pandas as pd
try:
    from IPython.display import display
except Exception:
    display = print

BASE_DIR = Path(os.environ.get("NIKL_PROJECT_DIR", ".")).resolve()
DATA_DIR = Path(os.environ.get("NIKL_PREPROCESS_OUT_DIR", str(BASE_DIR / "data" / "preprocessed_v3b"))).resolve()
REVIEW_FILE = os.environ.get("NIKL_REVIEW_FILE", "train.jsonl")     # 판정 대상 파일 (기본: train)
OUT_DIR = BASE_DIR / "output" / "flag_review"

SEED = 42
N_PER_FLAG = 30          # 플래그별 표본 수 (해당 행이 30개보다 적으면 전부)
FLAGS = [
    "flag_form_conflict_long",
    "flag_form_conflict",
    "flag_edit_large",
    "flag_punct_excess",
    "flag_digit_changed",
    "flag_alpha_changed",
    "flag_emoji_changed",
    "flag_partial_correction_suspect",
    "flag_missed_correction_suspect",
]
CONTROL = "CONTROL_no_flag"   # 통제군: 품질 플래그가 하나도 없는 행에서 뽑은 표본

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 250)
pd.set_option("display.unicode.east_asian_width", True)

assert (DATA_DIR / REVIEW_FILE).exists(), f"파일이 없습니다: {DATA_DIR / REVIEW_FILE}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("읽을 파일:", DATA_DIR / REVIEW_FILE)
print("결과 저장 폴더:", OUT_DIR)

읽을 파일: C:\Users\KDS-18\Desktop\pro4_team3\data\preprocessed_v3b\train.jsonl
결과 저장 폴더: C:\Users\KDS-18\Desktop\pro4_team3\output\flag_review


In [2]:
# 1. 파일을 한 줄씩 읽으며 플래그별 무작위 표본 뽑기 (저수지 표집, 메모리 절약)
import time
def load_conflicts(path):
    """train에서 만든 '같은 입력의 다른 정답들' 표 (참고용으로 표에 붙임)"""
    d = {}
    if path.exists():
        cf = pd.read_csv(path, encoding="utf-8-sig", dtype=str, keep_default_na=False)
        for r in cf.itertuples(index=False):
            d[r.input] = " | ".join(f"{t.strip()} ({c.strip()})" for t, c in zip(r.targets.split(" | "), r.counts.split(" | ")))
    return d
CONFLICTS = load_conflicts(DATA_DIR / "train_input_conflicts.csv")

keys = FLAGS + [CONTROL]
rngs = {k: random.Random(f"{SEED}-{k}") for k in keys}
seen = {k: 0 for k in keys}
reservoir = {k: [] for k in keys}
KEEP = ["utterance_id", "input", "target", "change_type", "quality_flags"]   # (n_quality_flags는 아래에서 직접 참조)

t0, n_rows = time.time(), 0
with open(DATA_DIR / REVIEW_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        r = json.loads(line); n_rows += 1
        qf = r["quality_flags"] or []
        hit = [k for k in FLAGS if k in qf] + ([CONTROL] if r["n_quality_flags"] == 0 else [])   # 통제군: 품질 플래그(정보용 제외)가 0개인 행
        row = {c: r[c] for c in KEEP}
        for k in hit:
            seen[k] += 1
            if len(reservoir[k]) < N_PER_FLAG:
                reservoir[k].append(row)
            else:
                j = rngs[k].randrange(seen[k])
                if j < N_PER_FLAG:
                    reservoir[k][j] = row
        if n_rows % 250_000 == 0:
            print(f"  ... {n_rows:,}행 읽음 ({time.time()-t0:.0f}초)")

TOTALS = dict(seen)
print(f"읽은 행: {n_rows:,} ({time.time()-t0:.0f}초)")
print(pd.DataFrame({"파일 전체에서 해당하는 행": TOTALS, "표본 수": {k: len(v) for k, v in reservoir.items()}}).to_string())

  ... 250,000행 읽음 (2초)
  ... 500,000행 읽음 (5초)
  ... 750,000행 읽음 (7초)
읽은 행: 986,728 (9초)
                                 파일 전체에서 해당하는 행  표본 수
flag_form_conflict_long                              17095       30
flag_form_conflict                                  116635       30
flag_edit_large                                      23458       30
flag_punct_excess                                    22165       30
flag_digit_changed                                    1372       30
flag_alpha_changed                                     236       30
flag_emoji_changed                                     376       30
flag_partial_correction_suspect                          6        6
flag_missed_correction_suspect                          23       23
CONTROL_no_flag                                     833318       30


## 판정 기준 (T / F)

**정책과 상관없이 "정답(target)에 명백한 결함이 있는가"만 봅니다.** 마침표·쉼표를 넣었다거나 `ㅇㅇ`를 `응응`으로 바꿨다는 것 자체는 결함이 아니라 *방침 문제*라서 F로 둡니다. (방침은 별도로 팀이 정합니다.)

| 판정 | 언제 |
|---|---|
| **T (문제 있음)** | 정답이 입력에 없던 내용을 덧붙이거나 지웠다 / 입력의 뜻이 바뀌었다 / 명백한 오타·띄어쓰기 오류가 정답에 그대로 남아 있다 / 정답이 깨진 문자열이다 |
| **F (정상)** | 띄어쓰기·철자·문장부호 교정이 타당하다, 또는 교정할 것이 없어서 그대로다 |
| **? (판단 불가)** | 초성체·은어라 맞는지 모르겠다 |

**판정 단위는 현재 표시된 행의 target입니다.**

`같은 입력의 다른 정답들`은 해당 플래그가 왜 발생했는지 이해하기 위한 참고 정보로만 사용합니다.

- 현재 표시된 target 자체에 명백한 결함이 있으면 T
- 현재 target이 타당하면 다른 후보 중 문제가 있어도 현재 행은 F
- 여러 target 간 불일치 자체가 문제라고 판단되는 경우에는 별도 메모로 기록하고, 현재 행의 T/F 판정과 구분합니다.

`CONTROL_no_flag`는 **플래그가 하나도 없는 행**의 표본입니다. 플래그가 붙은 행의 T 비율을 이것과 비교해야 "이 플래그가 실제로 문제를 골라내는지" 알 수 있습니다.

In [3]:
# 2. 표를 보여 주는 함수 (번호는 BAD / UNSURE에 적을 때 사용)
DESC = {
    "flag_form_conflict_long":
        "같은 입력(5자 이상)에 서로 다른 정답이 붙은 경우",

    "flag_form_conflict":
        "같은 입력(길이 무관)에 서로 다른 정답이 붙은 경우. 짧은 입력(ㅇㅇ, ㅋㅋ 등)이 많이 섞임",

    "flag_edit_large":
        "철자·단어 수준에서 편집량이 큰 경우",

    "flag_punct_excess":
        "문장부호가 3개 이상 늘었거나 같은 부호가 4번 이상 새로 생긴 경우",

    "flag_digit_changed":
        "숫자가 달라진 경우",

    "flag_alpha_changed":
        "영문이 달라진 경우",

    "flag_emoji_changed":
        "이모지·기호가 달라진 경우",

    "flag_partial_correction_suspect":
        "정답 안에 교정이 덜 된 부분이 남은 것으로 의심되는 경우",

    "flag_missed_correction_suspect":
        "train에서 거의 항상 고쳐지는 단어가 이 행에서는 고쳐지지 않은 경우"
}

def to_table(k):
    d = pd.DataFrame(reservoir[k])
    if d.empty:
        return d
    d.insert(0, "번호", range(1, len(d) + 1))
    d["입력"] = d["input"].str.replace("\r\n", "⏎", regex=False).str.replace("\n", "⏎", regex=False)
    d["정답"] = d["target"].str.replace("\r\n", "⏎", regex=False).str.replace("\n", "⏎", regex=False)
    d["같은 입력의 다른 정답들(빈도)"] = d["input"].map(CONFLICTS).fillna("")
    d["변경유형"] = d["change_type"]
    d["붙은 플래그"] = d["quality_flags"].map(lambda x: ", ".join(s.replace("flag_", "") for s in (x or [])))
    return d[["번호", "입력", "정답", "같은 입력의 다른 정답들(빈도)", "변경유형", "붙은 플래그"]]

def show(k):
    print(f"[{k}] 파일 전체 {TOTALS[k]:,}행 중 {len(reservoir[k])}개 표본" + (f" - {DESC[k]}" if k in DESC else " - 플래그가 없는 행(통제군)"))
    display(to_table(k))

### flag_form_conflict_long

In [4]:
show("flag_form_conflict_long")

[flag_form_conflict_long] 파일 전체 17,095행 중 30개 표본 - 같은 입력(5자 이상)에 서로 다른 정답이 붙은 경우


,번호,입력,정답,같은 입력의 다른 정답들(빈도),변경유형,붙은 플래그
0,1,그래서그런가,그래서 그런가?,그래서 그런가? (3) | 그래서 그런가 (1),punct_only,"form_conflict, form_conflict_long"
1,2,ㅋㅋㅋㅋㅋ,ㅋㅋㅋㅋㅋ,ㅋㅋㅋㅋㅋ (2027) | ㅋㅋㅋㅋㅋ. (1),unchanged,"form_conflict, form_conflict_long"
2,3,그러시군요,그러시군요.,"그러시군요. (18) | 그러시군요? (2) | 그러시군요, (1)",punct_only,"form_conflict, form_conflict_long"
3,4,안녕하세요,안녕하세요?,"안녕하세요? (743) | 안녕하세요. (181) | 안녕하세요 (20) | 안녕하세요, (7) | 안녕하세요 . (1) | 안녕하세요? ^ (1) | 안녕하세요. ㅎㅎ (1)",punct_only,"form_conflict, form_conflict_long"
4,5,ㅋㅋㅋㅋㅋㅋㅋㅋㅋㄴ,ㅋㅋㅋㅋㅋㅋㅋㅋㅋㄴ,ㅋㅋㅋㅋㅋㅋㅋㅋㅋㄴ (2) | ㅋㅋㅋㅋㅋㅋㅋㅋㅋ ㄴ (1),unchanged,"form_conflict, form_conflict_long"
5,6,ㅋㅋㅋㅋ아아,"ㅋㅋㅋㅋ 아아,","ㅋㅋㅋㅋ 아아, (1) | ㅋㅋㅋㅋ 아아. (1)",punct_only,"form_conflict, form_conflict_long"
6,7,앜ㅋㅋㅋㅋㅋㅋ,"아, ㅋㅋㅋㅋㅋㅋ","아. ㅋㅋㅋㅋㅋㅋㅋ (46) | 아, ㅋㅋㅋㅋㅋㅋㅋ (39) | 아, ㅋㅋㅋㅋㅋㅋ (5) | 앜, ㅋㅋㅋㅋㅋㅋ (1) | 아. ㅋㅋㅋㅋㅋㅋ (1) | 앜. ㅋㅋㅋㅋㅋㅋ (1) | 아. ㅋㅋㅋㅋㅋㅋㅋㅋ (1)",spelling_or_word,"form_conflict, form_conflict_long"
7,8,아 ㅋㅋㅋㅋ,아. ㅋㅋㅋㅋ,"아. ㅋㅋㅋㅋ (14) | 아, ㅋㅋㅋㅋ (12)",punct_only,"form_conflict, form_conflict_long"
8,9,할인받아서,할인받아서,할인받아서 (1) | 할인받아서. (1),unchanged,"form_conflict, form_conflict_long"
9,10,맞아요ㅠㅠㅠ,맞아요. ㅠㅠㅠ,"맞아요. ㅠㅠㅠ (20) | 맞아요, ㅠㅠㅠ (2)",punct_only,"form_conflict, form_conflict_long"


### flag_form_conflict

In [5]:
show("flag_form_conflict")

[flag_form_conflict] 파일 전체 116,635행 중 30개 표본 - 같은 입력(길이 무관)에 서로 다른 정답이 붙은 경우. 짧은 입력(ㅇㅇ, ㅋㅋ 등)이 많이 섞임


,번호,입력,정답,같은 입력의 다른 정답들(빈도),변경유형,붙은 플래그
0,1,와,"와,","와, (618) | 와. (113) | 와 (16) | 와! (10) | 와… (7) | 와... (6) | 왜? (2) | 와장창. (1)",punct_only,form_conflict
1,2,그치만..!,그렇지만!,그치만… (1) | 그렇지만! (1),spelling_or_word,form_conflict
2,3,아아 ㅎㅎ,아아. ㅎㅎ,"아아. ㅎㅎ (5) | 아아, ㅎㅎ (2)",punct_only,form_conflict
3,4,거의 뭐,"거의 뭐,","거의 뭐, (1) | 거의 뭐 (1)",punct_only,form_conflict
4,5,알겠,알겠어.,알겠지? (1) | 알겠어. (1),spelling_or_word,form_conflict
5,6,안녕하세요,안녕하세요.,"안녕하세요? (743) | 안녕하세요. (181) | 안녕하세요 (20) | 안녕하세요, (7) | 안녕하세요 . (1) | 안녕하세요? ^ (1) | 안녕하세요. ㅎㅎ (1)",punct_only,"form_conflict, form_conflict_long"
6,7,그건 맞아,그건 맞아,그건 맞아. (9) | 그건 맞아 (1),unchanged,form_conflict
7,8,호,호.,"호. (39) | 호, (9) | 호 (1)",punct_only,form_conflict
8,9,ㅋㅋ,ㅋㅋ,ㅋㅋ (1413) | ㅋㅋ. (8),unchanged,form_conflict
9,10,차라리,차라리,차라리 (18) | 차라리. (5),unchanged,form_conflict


### flag_edit_large

In [6]:
show("flag_edit_large")

[flag_edit_large] 파일 전체 23,458행 중 30개 표본 - 철자·단어 수준에서 편집량이 큰 경우


,번호,입력,정답,같은 입력의 다른 정답들(빈도),변경유형,붙은 플래그
0,1,돕다,덥다,,spelling_or_word,edit_large
1,2,ㅇㅇ 덥드라,"응응, 덥더라.",,spelling_or_word,edit_large
2,3,왐마,어머.,"어머. (4) | 왐마. (1) | 어머, (1)",spelling_or_word,"form_conflict, edit_large"
3,4,ㅇㅇ 맞아,응응. 맞아.,"응응. 맞아. (2) | 응응, 맞아. (2)",spelling_or_word,"form_conflict, edit_large"
4,5,눼,네.,"네. (16) | 눼, (3) | 눼. (1) | 네? (1)",spelling_or_word,"form_conflict, edit_large"
5,6,ㅂ두유??,주유?,,spelling_or_word,edit_large
6,7,몬데몬데,"뭔데, 뭔데?","뭔데 뭔데? (1) | 뭔데, 뭔데? (1)",spelling_or_word,"form_conflict, edit_large"
7,8,호 ㄱㅅㄱㅅ,"호, 감사감사.",,spelling_or_word,edit_large
8,9,않이,아니.,"많이 (1) | 아니, (1) | 아니. (1) | 아니 (1)",spelling_or_word,"form_conflict, edit_large"
9,10,꼬소혀,고소해.,,spelling_or_word,edit_large


### flag_punct_excess

In [7]:
show("flag_punct_excess")

[flag_punct_excess] 파일 전체 22,165행 중 30개 표본 - 문장부호가 3개 이상 늘었거나 같은 부호가 4번 이상 새로 생긴 경우


,번호,입력,정답,같은 입력의 다른 정답들(빈도),변경유형,붙은 플래그
0,1,아 드림은 정말 어렸을때 데뷔하긴 했어요ㅎㅎ 근데 저도 사실 엔시티 체제를 아직 제대로 이해못했어요ㅋㅋ,"아, 드림은 정말 어렸을 때 데뷔하긴 했어요. ㅎㅎ 근데 저도 사실 엔시티 체제를 아직 제대로 이해 못 했어요. ㅋㅋ",,punct_only,punct_excess
1,2,ㅎㅎㅎ이미 저의 모 업무담당자님은 전 부서에서 그분은 왜 대화가 안통하시죠 라는 분이라서,"ㅎㅎㅎ 이미 저의 모 업무 담당자님은 전 부서에서 ""그분은 왜 대화가 안 통하시죠?""라는 분이라서",,punct_only,punct_excess
2,3,하면? 스트레스를 덜 받는다..,하면(?) 스트레스를 덜 받는다...,,punct_only,punct_excess
3,4,오 난 그 뭐지 아바타 배경,"오, 난 그 뭐지? 아바타 배경.",,punct_only,punct_excess
4,5,월세였으면 안나갔을듯 ㅠㅠ 코로나때문에 빈방 넘친다하더라고,"월세였으면 안 나갔을 듯, ㅠㅠ ""코로나 때문에 빈방 넘친다."" 하더라고.",,punct_only,punct_excess
5,6,지금 44된거보면 휴 감지덕지,"지금 44된 거 보면, 휴, 감지덕지.",,punct_only,punct_excess
6,7,아 미치겟다 너무 배불러,"아, 미치겠다, 너무 배불러.",,spelling_or_word,punct_excess
7,8,아 제 머리도 얼른 묶일 정도만큼은 길었으면 좋겠어요.. 애매해서 너무 귀찮아요 ㅋㅋㅋ,"아, 제 머리도 얼른 묶일 정도만큼은 길었으면 좋겠어요... 애매해서 너무 귀찮아요. ㅋㅋㅋ",,punct_only,punct_excess
8,9,오 다행이다 그건,"오, 다행이다, 그건.",,punct_only,punct_excess
9,10,초읍가주세요 하자마자,"""초읍 가주세요."" 하자마자",,punct_only,punct_excess


### flag_digit_changed

In [8]:
show("flag_digit_changed")

[flag_digit_changed] 파일 전체 1,372행 중 30개 표본 - 숫자가 달라진 경우


,번호,입력,정답,같은 입력의 다른 정답들(빈도),변경유형,붙은 플래그
0,1,옼0,옼.,,spelling_or_word,"edit_large, digit_changed"
1,2,지금은 또 1도 안하네여,지금은 또 하나도 안 하네요.,,spelling_or_word,digit_changed
2,3,4계절이 있어서 좋기도한데 그만큼 계절이 지나가는게 아쉽네요ㅠㅠ,사계절이 있어서 좋기도 한데 그만큼 계절이 지나가는 게 아쉽네요. ㅠㅠ,,spelling_or_word,digit_changed
3,4,ㅕ못하는고 1도없구요.ㅡ,못하는 거 하나도 없고요.,,spelling_or_word,"edit_large, digit_changed"
4,5,후회하는 애들 1도 없음?,후회하는 애들 하나도 없음?,,spelling_or_word,digit_changed
5,6,무상의 물 처음에 웬 슬라임을 소환하길ㄹ9,무상의 물 처음에 웬 슬라임을 소환하길래,,spelling_or_word,digit_changed
6,7,"일회용컵도 쥴이고, 설거지하느라 사용하는 세제양도 줗이고 1석 2조네요!","일회용 컵도 줄이고, 설거지하느라 사용하는 세제 양도 줄이고 일석이조네요!",,spelling_or_word,digit_changed
7,8,저 내일 반차인데 이번주 반차쓰는거 설레서 일 1개도안했어요,저 내일 반차인데 이번 주 반차 쓰는 거 설레서 일 한 개도 안 했어요.,,spelling_or_word,digit_changed
8,9,계란도 며칠전에 한판 7500원에 샀더니 일주일 뒤 6500원.. ㅠㅠ,"계란도 며칠 전에 한 판 7,500원에 샀더니 일주일 뒤 6,500원... ㅠㅠ",,punct_only,"digit_changed, punct_excess"
9,10,일까지는,8일까지는,,spelling_or_word,digit_changed


### flag_alpha_changed

In [9]:
show("flag_alpha_changed")

[flag_alpha_changed] 파일 전체 236행 중 30개 표본 - 영문이 달라진 경우


,번호,입력,정답,같은 입력의 다른 정답들(빈도),변경유형,붙은 플래그
0,1,자기가 12시 퇴근m,자기가 12시 퇴근?,,spelling_or_word,alpha_changed
1,2,B플은,비플은,,spelling_or_word,alpha_changed
2,3,좋아yo,좋아요.,,spelling_or_word,"edit_large, alpha_changed"
3,4,디지털 CD로 샀었나?,디지털 시디로 샀었나?,,spelling_or_word,alpha_changed
4,5,da른 친구들,다른 친구들,,spelling_or_word,alpha_changed
5,6,mri를 60만원 주고 찍었는데,엠아르아이를 60만 원 주고 찍었는데,,spelling_or_word,alpha_changed
6,7,무드등 같은건가유 예브답,무드등 같은 건_x0008_가요? 예쁘다.,,spelling_or_word,"digit_changed, alpha_changed"
7,8,뉴스보니까 화이자가 fda정식 승인 났다고 하더라고요,뉴스 보니까 화이자가 에프디에이 정식 승인 났다고 하더라고요.,,spelling_or_word,alpha_changed
8,9,요즘 좋아하는 tv 프로그램 있어요?,요즘 좋아하는 티브이 프로그램 있어요?,,spelling_or_word,alpha_changed
9,10,iluvu,I luv u,,spelling_or_word,alpha_changed


### flag_emoji_changed

In [10]:
show("flag_emoji_changed")

[flag_emoji_changed] 파일 전체 376행 중 30개 표본 - 이모지·기호가 달라진 경우


,번호,입력,정답,같은 입력의 다른 정답들(빈도),변경유형,붙은 플래그
0,1,ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋname5는 수술함;;;;;;; 말해도 되나ㅎ;;;;;,ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ name5는 수술함... 말해도 되나. ㅎ;,,punct_only,emoji_changed
1,2,헐;;;;우우쨌길래 그랬노,헐. 어쨌길래 그랬노?,,spelling_or_word,emoji_changed
2,3,잘자요 내사랑️️️,"잘 자요, 내 사랑.",,spelling_or_word,"emoji_changed, len_shrink"
3,4,아가야️,"아가야,",,spelling_or_word,emoji_changed
4,5,굶는거는 도저히 안되서;;;,굶는 거는 도저히 안돼서...,,spelling_or_word,emoji_changed
5,6,뭘 피격 횟수까지 카운팅해;;,뭘 피격 횟수까지 카운팅 해…,,punct_only,emoji_changed
6,7,상관있을듯 ;;;;,상관있을 듯.,,punct_only,emoji_changed
7,8,코로나 전처럼 바글바글하더라구요;;,코로나 전처럼 바글바글하더라고요…,,spelling_or_word,emoji_changed
8,9,바로 나냐고 ;; 어이업서,바로 나냐고… 어이없어.,,spelling_or_word,emoji_changed
9,10,진짜야;;;,진짜야…,,punct_only,emoji_changed


### flag_partial_correction_suspect

In [11]:
show("flag_partial_correction_suspect")

[flag_partial_correction_suspect] 파일 전체 6행 중 6개 표본 - 정답 안에 교정이 덜 된 부분이 남은 것으로 의심되는 경우


,번호,입력,정답,같은 입력의 다른 정답들(빈도),변경유형,붙은 플래그
0,1,안뇽,안뇽?,"안녕? (16) | 안녕. (8) | 안뇽? (3) | 안녕 (2) | 안녕, (1)",punct_only,"form_conflict, partial_correction_suspect"
1,2,ㅇㅋ. .. .,ㅇㅋ...,,punct_only,partial_correction_suspect
2,3,ㅇㅈ....,ㅇㅈ...,인정… (1) | ㅇㅈ... (1) | 인정... (1),punct_only,"form_conflict, partial_correction_suspect"
3,4,시상에,시상에.,"세상에, (32) | 세상에. (13) | 세상에! (5) | 세상에 (2) | 시상에. (1)",punct_only,"form_conflict, partial_correction_suspect"
4,5,안뇽,안뇽?,"안녕? (16) | 안녕. (8) | 안뇽? (3) | 안녕 (2) | 안녕, (1)",punct_only,"form_conflict, partial_correction_suspect"
5,6,안뇽,안뇽?,"안녕? (16) | 안녕. (8) | 안뇽? (3) | 안녕 (2) | 안녕, (1)",punct_only,"form_conflict, partial_correction_suspect"


### flag_missed_correction_suspect

In [12]:
show("flag_missed_correction_suspect")

[flag_missed_correction_suspect] 파일 전체 23행 중 23개 표본 - train에서 거의 항상 고쳐지는 단어가 이 행에서는 고쳐지지 않은 경우


,번호,입력,정답,같은 입력의 다른 정답들(빈도),변경유형,붙은 플래그
0,1,ㅇㅇㅇㅇ,ㅇㅇㅇㅇ,"응응응응, (111) | 응응응응. (110) | 응응응응! (5) | ㅇㅇㅇㅇ (4) | 응응응, (1) | 응응, 응응. (1)",unchanged,"form_conflict, missed_correction_suspect"
1,2,ㅇㅇㅇㅇㅇ,ㅇㅇㅇㅇㅇ,"응응응응응. (56) | 응응응응응, (35) | ㅇㅇㅇㅇㅇ (6) | 응응응응응! (3)",unchanged,"form_conflict, form_conflict_long, missed_correction_suspect"
2,3,ㅇㅇㅇㅇ,ㅇㅇㅇㅇ,"응응응응, (111) | 응응응응. (110) | 응응응응! (5) | ㅇㅇㅇㅇ (4) | 응응응, (1) | 응응, 응응. (1)",unchanged,"form_conflict, missed_correction_suspect"
3,4,ㅇㅇㅇㅇㅇ,ㅇㅇㅇㅇㅇ,"응응응응응. (56) | 응응응응응, (35) | ㅇㅇㅇㅇㅇ (6) | 응응응응응! (3)",unchanged,"form_conflict, form_conflict_long, missed_correction_suspect"
4,5,ㅋㅋ ㅇㅋ,ㅋㅋ ㅇㅋ,,unchanged,missed_correction_suspect
5,6,ㅇㅋㅇㅋ,ㅇㅋㅇㅋ,"오키오키. (179) | 오키오키, (19) | 오키, 오키. (4) | ㅇㅋㅇㅋ (3) | 오키오키! (3) | 오키오키 (2) | 욐오키. (1)",unchanged,"form_conflict, missed_correction_suspect"
6,7,ㅇㅋㅇㅋ,ㅇㅋㅇㅋ,"오키오키. (179) | 오키오키, (19) | 오키, 오키. (4) | ㅇㅋㅇㅋ (3) | 오키오키! (3) | 오키오키 (2) | 욐오키. (1)",unchanged,"form_conflict, missed_correction_suspect"
7,8,ㅇㅋ,ㅇㅋ,"오키. (311) | 오키, (51) | 오키 (3) | ㅇㅋ (3) | 오키... (1) | 오키! (1)",unchanged,"form_conflict, missed_correction_suspect"
8,9,ㅇㅋ,ㅇㅋ,"오키. (311) | 오키, (51) | 오키 (3) | ㅇㅋ (3) | 오키... (1) | 오키! (1)",unchanged,"form_conflict, missed_correction_suspect"
9,10,ㅇㅋ,ㅇㅋ,"오키. (311) | 오키, (51) | 오키 (3) | ㅇㅋ (3) | 오키... (1) | 오키! (1)",unchanged,"form_conflict, missed_correction_suspect"


### CONTROL_no_flag

In [13]:
show("CONTROL_no_flag")

[CONTROL_no_flag] 파일 전체 833,318행 중 30개 표본 - 플래그가 없는 행(통제군)


,번호,입력,정답,같은 입력의 다른 정답들(빈도),변경유형,붙은 플래그
0,1,생각보다 진입장벽이 높지 않아요,생각보다 진입 장벽이 높지 않아요.,,punct_only,
1,2,표면이 까매 져,표면이 까매져.,,punct_only,
2,3,5시 도착?,5시 도착?,,unchanged,
3,4,돈 벌어서 엉뚱한데.. 쓰는....,돈 벌어서 엉뚱한데 쓰는,,punct_only,
4,5,아 그건 모르겠어요 ㅋㅋㅋㅋㅋㅋ,"아, 그건 모르겠어요. ㅋㅋㅋㅋㅋㅋ",,punct_only,
5,6,"회사 다니는게 더 고생이 많으시죠!ㅎㅎ 지금 무슨 일 하시는지, 얼마나 오래 하셨는지 여쭤봐도 될까요?","회사 다니는 게 더 고생이 많으시죠! ㅎㅎ 지금 무슨 일 하시는지, 얼마나 오래 하셨는지 여쭤봐도 될까요?",,spacing_only,
6,7,요즘 부쩍 체력이 떨어지고 있어서 야외활동이 적네요,"요즘 부쩍 체력이 떨어지고 있어서, 야외 활동이 적네요.",,punct_only,
7,8,저 화장 10분만에 할 수 있어요,저 화장 10분 만에 할 수 있어요.,,punct_only,
8,9,저도 나름 말하는감자 거든요,저도 나름 말하는 감자거든요.,,punct_only,
9,10,불편하다고 다 그랬으면서,불편하다고 다 그랬으면서.,,punct_only,


## 판정 입력

위 표를 하나씩 확인한 뒤 판정합니다.

### T인 경우

`BAD`에 아래처럼 **번호와 이유를 함께 기록**합니다.

```python
"flag_digit_changed": {
    4: "숫자 정보가 원문과 다르게 변경됨",
    18: "원문의 숫자가 정답에서 삭제됨"
}

```
### ?인 경우
판단하기 어려운 번호는 UNSURE에 기록합니다.

```python
"flag_digit_changed": [7, 21]
```

### T가 하나도 없는 경우
30개 표본을 모두 확인했지만 문제가 하나도 없다면
NONE_BAD에 플래그 이름을 추가합니다.

```python
NONE_BAD = [
    "flag_punct_excess"
]
```

#### 빈 딕셔너리 {} 또는 빈 리스트 [] 상태는 아직 판정하지 않은 상태로 간주합니다.


In [14]:
# 3. 판정 입력 (T 번호와 판정 이유 기록)

BAD = {

    "flag_form_conflict_long": {
        # 명백한 맞춤법·문법 오류 없음
    },

    "flag_form_conflict": {
        # 명백한 맞춤법·문법 오류 없음
    },

    "flag_edit_large": {
        # 현재 확인한 표본 중 확정 BAD 없음
    },

    "flag_punct_excess": {
        # 반복 문장부호/감정 표현은 이번 프로젝트의 핵심 교정 범위에서 제외
    },

    "flag_digit_changed": {
        20: "'고시생1..'을 '고시생 하나...'로 변경하여 숫자 1을 임의로 자연어 표현으로 해석한 오류",
    },

    "flag_alpha_changed": {
        7: "target에 '_x0008_' 제어문자 표현이 남아 있어 명백한 데이터/인코딩 오류",
        11: "'퐁당퐁당러브'가 target에서 '퐁당퐁당 lovr'로 변경되어 잘못된 문자열이 생성됨",
        23: "'ㄱㄱ'가 target에서 'Rr'로 변경되어 의미 없는 영문 문자열이 생성됨",
        28: "URL 인코딩 형태의 입력에 대해 target이 'ERROR:#REF!'로 저장되어 명백한 정답 데이터 오류",
    },

    "flag_emoji_changed": {
        16: "입력의 '.)'를 target에서 ':)'로 변경하여 원문에 없던 이모티콘 형태를 추가함",
    },

    "flag_partial_correction_suspect": {
        1: "'안뇽'이 target에도 그대로 남아 있어 '안녕'으로의 명백한 표기 교정이 누락됨",
        4: "'시상에'가 target에도 그대로 남아 있어 '세상에'로의 명백한 표기 교정이 누락됨",
        5: "'안뇽'이 target에도 그대로 남아 있어 '안녕'으로의 명백한 표기 교정이 누락됨",
        6: "'안뇽'이 target에도 그대로 남아 있어 '안녕'으로의 명백한 표기 교정이 누락됨",
    },

    "flag_missed_correction_suspect": {
        # 표본 23개가 모두 ㅇㅇ, ㅇㅋ, ㅇㅈ 등 채팅 축약/반복 표현이므로
        # 이번 프로젝트의 핵심 교정 범위에서 제외
    },

    "CONTROL_no_flag": {
        # 현재 표본에서는 명백한 오류 없음
    },
}


UNSURE = {

    "flag_form_conflict_long": [],

    "flag_form_conflict": [],

    "flag_edit_large": [],

    "flag_punct_excess": [],

    "flag_digit_changed": [],

    "flag_alpha_changed": [],

    "flag_emoji_changed": [],

    "flag_partial_correction_suspect": [
        # 2: 'ㅇㅋ...' → 채팅 축약어이므로 F
        # 3: 'ㅇㅈ...' → 채팅 축약어이므로 F
    ],

    "flag_missed_correction_suspect": [
        # 전부 F 처리
    ],

    "CONTROL_no_flag": [],
}


NONE_BAD = [
    "flag_form_conflict",
    "flag_form_conflict_long",
    "flag_edit_large",
    "flag_punct_excess",
    "flag_missed_correction_suspect",
    "CONTROL_no_flag",
]

In [15]:
# 4. 결과 계산: 플래그별 T 비율과 95% 신뢰구간(Wilson)
def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (max(0.0, c - h), min(1.0, c + h))

rows = []
for k in FLAGS + [CONTROL]:
    n = len(reservoir[k])

    bad = set(BAD.get(k, {}).keys())
    uns = set(UNSURE.get(k, []))
    
    for name, s in (("BAD", bad), ("UNSURE", uns)):
        wrong = [i for i in s if not (1 <= i <= n)]
        assert not wrong, f"{k}의 {name}에 범위(1~{n})를 벗어난 번호가 있습니다: {wrong}"
    assert not (bad & uns), f"{k}: 같은 번호가 BAD와 UNSURE에 모두 있습니다: {sorted(bad & uns)}"
    judged = bool(bad or uns or k in NONE_BAD)
    if not judged or n == 0:
        rows.append({"대상": k, "파일 전체 행 수": TOTALS[k], "표본": n, "T": None, "F": None, "?": None,
                     "T 비율(%)": None, "95% 신뢰구간(%)": "미판정", "지우면 버려지는 정상 행 추정(건)": ""})
        continue
    t, u = len(bad), len(uns); f_ = n - t - u
    lo, hi = wilson(t, t + f_)
    r = {"대상": k, "파일 전체 행 수": TOTALS[k], "표본": n, "T": t, "F": f_, "?": u,
         "T 비율(%)": round(t / (t + f_) * 100, 1) if (t + f_) else None,
         "95% 신뢰구간(%)": f"{lo*100:.0f} ~ {hi*100:.0f}"}
    r["지우면 버려지는 정상 행 추정(건)"] = "" if k == CONTROL else f"{int(TOTALS[k]*(1-hi)):,} ~ {int(TOTALS[k]*(1-lo)):,}"
    rows.append(r)
RESULT = pd.DataFrame(rows)
display(RESULT)

,대상,파일 전체 행 수,표본,T,F,?,T 비율(%),95% 신뢰구간(%),지우면 버려지는 정상 행 추정(건)
0,flag_form_conflict_long,17095,30,0,30,0,0.0,0 ~ 11,"15,154 ~ 17,095"
1,flag_form_conflict,116635,30,0,30,0,0.0,0 ~ 11,"103,394 ~ 116,635"
2,flag_edit_large,23458,30,0,30,0,0.0,0 ~ 11,"20,795 ~ 23,458"
3,flag_punct_excess,22165,30,0,30,0,0.0,0 ~ 11,"19,648 ~ 22,165"
4,flag_digit_changed,1372,30,1,29,0,3.3,1 ~ 17,"1,143 ~ 1,363"
5,flag_alpha_changed,236,30,4,26,0,13.3,5 ~ 30,165 ~ 223
6,flag_emoji_changed,376,30,1,29,0,3.3,1 ~ 17,313 ~ 373
7,flag_partial_correction_suspect,6,6,4,2,0,66.7,30 ~ 90,0 ~ 4
8,flag_missed_correction_suspect,23,23,0,23,0,0.0,0 ~ 14,19 ~ 23
9,CONTROL_no_flag,833318,30,0,30,0,0.0,0 ~ 11,


## 결과 읽는 법

- **T 비율**은 그 플래그가 붙은 행 중 사람이 보기에 실제로 문제인 행의 비율(표본 추정)입니다. 표본이 30개라 신뢰구간이 넓습니다(예: 30개 중 3개가 T이면 대략 4~26%). 구간이 넓으면 결론을 단정하지 마세요.
- **통제군(플래그 없음)의 T 비율과 비교**하세요. 플래그 행의 T 비율이 통제군과 비슷하면 그 플래그는 문제를 골라내지 못하는 것입니다.
- **"지우면 버려지는 정상 행 추정"**이 크면, 그 플래그로 학습 행을 삭제하는 것은 손해입니다. 삭제 대신 정보용으로만 두거나, 특정 조건과 결합한 규칙을 다시 만들어야 합니다.
- 이 결과는 **이 표본을 판정한 사람의 기준**에 따른 값입니다. 판정자가 여러 명이면 같은 표본을 서로 따로 판정해 일치도를 함께 보고하는 것이 좋습니다.
- 표본에서 T를 판정한 이유(예: "정답에서 내용이 삭제됨")를 보고서에 사례로 함께 적어 두세요.
- 각 품질 플래그는 서로 배타적이지 않으며 하나의 행에 여러 플래그가 동시에 붙을 수 있습니다. 따라서 플래그별 T 건수를 단순 합산해 전체 오류 건수로 해석하지 않습니다.

In [16]:
# 5. 판정 결과 저장

# 아직 판정하지 않은 대상 확인
UNJUDGED_FLAGS = [
    k for k in FLAGS + [CONTROL]
    if len(reservoir[k]) > 0
    and not (
        len(BAD.get(k, {})) > 0
        or len(UNSURE.get(k, [])) > 0
        or k in NONE_BAD
    )
]

print("미판정 대상:", UNJUDGED_FLAGS)

if UNJUDGED_FLAGS:
    print("\n아직 판정하지 않은 대상이 있으므로 최종 CSV 저장은 하지 않습니다.")
    print("모든 표본을 확인한 뒤 BAD / UNSURE / NONE_BAD를 작성하세요.")

else:
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    label_rows = []

    for k in FLAGS + [CONTROL]:

        bad_dict = BAD.get(k, {})
        unsure_set = set(UNSURE.get(k, []))

        for i, r in enumerate(reservoir[k], start=1):

            if i in bad_dict:
                lab = "T"
                reason = bad_dict[i]

            elif i in unsure_set:
                lab = "?"
                reason = "판단 불가"

            else:
                lab = "F"
                reason = ""

            label_rows.append({
                "대상": k,
                "번호": i,
                "판정": lab,
                "판정이유": reason,
                "utterance_id": r["utterance_id"],
                "input": r["input"],
                "target": r["target"],
            })

    labels_path = OUT_DIR / f"flag_review_labels_{stamp}.csv"
    result_path = OUT_DIR / f"flag_review_result_{stamp}.csv"

    pd.DataFrame(label_rows).to_csv(
        labels_path,
        index=False,
        encoding="utf-8-sig"
    )

    RESULT.to_csv(
        result_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("저장:", labels_path.name)
    print("저장:", result_path.name)

미판정 대상: []
저장: flag_review_labels_20260920_163018.csv
저장: flag_review_result_20260920_163018.csv
